# VLA Simulation — Google Colab (무료 GPU) 스타터

노트북(GPU) 없는 학생용. **런타임 → 런타임 유형 변경 → GPU(T4)** 선택 후 위에서부터 실행.

- 무거운 VLA(Qwen3-VL-2B) 추론을 Colab 무료 T4 GPU에서 실행
- Gazebo 화면은 **Xvfb + noVNC**로 브라우저에서 확인
- 사전 학습 산출물은 **HuggingFace Hub에서 import**(다운로드 관리 자동)

> ⚠️ 스타터 템플릿입니다. Gazebo Classic + ROS2 Humble를 Colab(Ubuntu 22.04)에 올리는 과정은
> 시간이 걸리고 환경에 따라 조정이 필요합니다. 셀별 주석을 따라 조정하세요.

## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch; print('CUDA:', torch.cuda.is_available())

## 1. 저장소 클론

In [ ]:
%cd /content
!git clone https://github.com/SKKUAutoLab/VLA_Simulation.git ros2_autonomous_vehicle_simulation
%cd /content/ros2_autonomous_vehicle_simulation

## 2. ROS 2 Humble + Gazebo Classic 설치 (Colab = Ubuntu 22.04)

In [ ]:
# ROS2 apt 소스 추가 후 ros-humble-desktop + gazebo 설치 (수 분 소요)
!apt-get update -q && apt-get install -y -q software-properties-common curl gnupg lsb-release
!curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg
!echo "deb [signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu jammy main" > /etc/apt/sources.list.d/ros2.list
!apt-get update -q && apt-get install -y -q ros-humble-ros-base ros-humble-gazebo-ros-pkgs gazebo
!pip install -q transformers peft accelerate huggingface_hub

## 3. 사전 학습 산출물 — HuggingFace Hub에서 import (데이터·학습 생략)

In [ ]:
import shutil
from huggingface_hub import hf_hub_download, snapshot_download
REPO = 'hoonsy/VLA_Simulation-pretrained'   # 실제 배포 repo로 교체
DEST = '/content/ros2_autonomous_vehicle_simulation/lora_pipeline'
for f in ('vla_lora_head_fast.pt', 'vla_lora_head.pt'):
    shutil.copy(hf_hub_download(repo_id=REPO, filename=f), f'{DEST}/{f}')
d = snapshot_download(repo_id=REPO, allow_patterns='vla_lora_adapter/*')
shutil.copytree(f'{d}/vla_lora_adapter', f'{DEST}/vla_lora_adapter', dirs_exist_ok=True)
print('산출물 로드 완료 — 데이터 수집·학습 없이 추론 가능')

## 4. Gazebo를 브라우저에서 보기 — Xvfb + noVNC
실행 후 출력되는 noVNC 링크를 클릭하면 브라우저에서 Gazebo 3D 화면을 볼 수 있습니다.

In [ ]:
!apt-get install -y -q xvfb x11vnc websockify novnc >/dev/null
import os, subprocess, time
os.environ['DISPLAY'] = ':1'
subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1280x800x24'])
time.sleep(2)
subprocess.Popen(['x11vnc', '-display', ':1', '-nopw', '-forever', '-shared', '-rfbport', '5900'])
subprocess.Popen(['websockify', '--web=/usr/share/novnc', '6080', 'localhost:5900'])
from google.colab.output import eval_js
print('noVNC:', eval_js('google.colab.kernel.proxyPort(6080)') + 'vnc.html')

## 5. 빌드 & VLA 주행 실행 (Xvfb 디스플레이 → noVNC로 관찰)

In [ ]:
%%bash
source /opt/ros/humble/setup.bash
cd /content/ros2_autonomous_vehicle_simulation
colcon build --packages-select interfaces_pkg --allow-overriding interfaces_pkg && source install/local_setup.bash
colcon build --symlink-install --packages-select simulation_pkg camera_perception_pkg lidar_perception_pkg qwen_vl_pkg
source install/local_setup.bash
# 주행 (noVNC 탭에서 Gazebo 확인)
DISPLAY=:1 ros2 launch lora_pipeline/vla_drive.launch.py gzclient:=true